In [ ]:
import lgdo
import lh5
import numpy as np
import awkward as ak
import os
import re

In [ ]:
filepath = "/mnt/atlas02/users/leoli/scarf/sp01/data/hit_vcs"
files = os.listdir(filepath)
files.sort()
files

pnum_cal = "00"
run_cal = "037"

pnum_phy_bg = "00"
run_phy_bg = "038"

pnum_phy_signal = "05"
run_phy_signal = "059"

runs = [
    's60ns_mw1',  's60ns_mw3',  's60ns_mw5',  's60ns_mw7',
    's100ns_mw1', 's100ns_mw3', 's100ns_mw5', 's100ns_mw7',
    's200ns_mw1', 's200ns_mw3', 's200ns_mw5', 's200ns_mw7',
    's300ns_mw1', 's300ns_mw3', 's300ns_mw5', 's300ns_mw7','s300ns_mw15',
    's500ns_mw1', 's500ns_mw3', 's500ns_mw5', 's500ns_mw7','s500ns_mw15',
    's700ns_mw1', 's700ns_mw3', 's700ns_mw5', 's700ns_mw7','s700ns_mw15',
    's1000ns_mw1',  's1000ns_mw3',  's1000ns_mw5',  's1000ns_mw7','s1000ns_mw15',
    's2000ns_mw3',  's2000ns_mw5',  's2000ns_mw7',
]

def build_run_db(runs, filepath):
    db = {}
    missing = []

    for run in runs:
        m = re.match(r"s(\d+)ns_mw(\d+)", run)
        if not m:
            missing.append(f"{run}: invalid run name format")
            continue

        run_path = os.path.join(filepath, run)
        phy_bg_path = os.path.join(run_path, "phy", f"p{pnum_phy_bg}", f"r{run_phy_bg}")
        phy_signal_path = os.path.join(run_path, "phy", f"p{pnum_phy_signal}", f"r{run_phy_signal}")
        cal_path = os.path.join(run_path, "cal", f"p{pnum_cal}", f"r{run_cal}")

        missing_this_run = []

        if not os.path.exists(run_path):
            missing_this_run.append(f"run folder missing: {run_path}")
        if not os.path.exists(phy_bg_path):
            missing_this_run.append(f"phy_bg path missing: {phy_bg_path}")
        if not os.path.exists(phy_signal_path):
            missing_this_run.append(f"phy_signal path missing: {phy_signal_path}")
        if not os.path.exists(cal_path):
            missing_this_run.append(f"cal path missing: {cal_path}")

        if missing_this_run:
            missing.append(f"{run}\n  " + "\n  ".join(missing_this_run))
            continue

        phy_bg_files = sorted(os.listdir(phy_bg_path))
        phy_signal_files = sorted(os.listdir(phy_signal_path))
        cal_files = sorted(os.listdir(cal_path))

        db[run] = {
            "run": run,
            "interval": f"{m.group(1)}ns",
            "mw": f"mw{m.group(2)}",

            "phy_bg_path": phy_bg_path,
            "phy_signal_path": phy_signal_path,
            "cal_path": cal_path,

            "phy_bg_files": phy_bg_files,
            "phy_signal_files": phy_signal_files,
            "cal_files": cal_files,

            "phy_bg_pathlist": [os.path.join(phy_bg_path, f) for f in phy_bg_files],
            "phy_signal_pathlist": [os.path.join(phy_signal_path, f) for f in phy_signal_files],
            "cal_pathlist": [os.path.join(cal_path, f) for f in cal_files],
        }

    if missing:
        raise FileNotFoundError(
            "[ERROR] Some runs or paths are missing:\n\n" + "\n\n".join(missing)
        )

    return db
run_db = build_run_db(runs, filepath)

In [ ]:
E_std=lh5.read("ch002/hit/trapEftp_fix_cal",run_db['s100ns_mw5']["phy_signal_pathlist"])
A_max_mw_o_E_Classifier_std=lh5.read("ch002/hit/A_max_mw_o_E_Classifier",run_db['s100ns_mw5']["phy_signal_pathlist"])
Iasy_std = lh5.read("ch002/hit/Iasy",run_db['s100ns_mw5']["phy_signal_pathlist"])
is_valid_waveform_std = lh5.read("ch002/hit/is_valid_waveform",run_db['s100ns_mw5']["phy_signal_pathlist"])
A_max_mw_o_E_High_Side_Cut_std = lh5.read("ch002/hit/A_max_mw_o_E_High_Side_Cut",run_db['s100ns_mw5']["phy_signal_pathlist"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pygama.pargen.energy_cal as pgc
import pygama.pargen.survival_fractions as sf

In [ ]:
FIT_CONFIGS = {
    "dep": {
        "range": (1565.0, 1615.0),
        "peak": 1592.5,
        "func": sf.gauss_on_step,
    },
    "sep": {
        "range": (2070.0, 2140.0),
        "peak": 2103.5,
        "func": sf.hpge_peak,
    },
    "fep": {
        "range": (2580.0, 2645.0),
        "peak": 2614.5,
        "func": sf.hpge_peak,
    },
}

In [ ]:
run_1 = "s60ns_mw3"
run_2 = "s300ns_mw5"

In [ ]:
pathlist = "phy_signal_pathlist"

In [ ]:
E_1=lh5.read("ch002/hit/trapEftp_fix_cal",run_db[run_1][pathlist])
A_max_mw_o_E_Classifier_1=lh5.read("ch002/hit/A_max_mw_o_E_Classifier",run_db[run_1][pathlist])
Iasy_std = lh5.read("ch002/hit/Iasy",run_db['s100ns_mw5'][pathlist])
is_valid_waveform_1= lh5.read("ch002/hit/is_valid_waveform",run_db['s100ns_mw5'][pathlist])
low_cut_1 = lh5.read("ch002/hit/A_max_mw_o_E_fix_Low_Cut",run_db['s100ns_mw5'][pathlist])

In [ ]:
E_1 = np.asarray(E_1)
# A_max_mw_o_E_High_Side_Cut_std = np.asarray(A_max_mw_o_E_High_Side_Cut_std)
Iasy_std = np.asarray(Iasy_std)
A_max_mw_o_E_Classifier_1= np.asarray(A_max_mw_o_E_Classifier_1)
is_valid_waveform_1 = np.asarray(is_valid_waveform_1)
low_cut_1 = np.asarray(low_cut_1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec

# ----------------------------
# 1. Prepare aligned data
# ----------------------------
x_all = np.asarray(A_max_mw_o_E_Classifier_1)
y_all = np.asarray(A_max_mw_o_E_Classifier_2)
E_all = np.asarray(E)

assert len(x_all) == len(y_all) == len(E_all)

finite = (
    np.isfinite(x_all)
    & np.isfinite(y_all)
    & np.isfinite(E_all)
)

# Choose the population to study:
# DEP window example
selection = (
    finite
    & valid
    & (E_all > 1585)
    & (E_all < 1600)
)

x = x_all[selection]
y = y_all[selection]

print("Selected events:", len(x))